# Notebook for computing hashes, buckets and similarity values for the disk scheme using a hybrid approach

Utilizes the disk scheme

Incorporates:
* Hashing of trajectories using disk scheme
* Bucketing of hashes made from disk scheme
* Similarity computation between trajectories within buckets.
    * Both for DTW and Frechet
* Analysis of the produced bucket system

Produces:
* JSON file containing buckets
* Similarity values for trajectories within buckets


## Hybrid approach

In [ ]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from computation.similarity import *
from utils.helpers.bucket_evaluation import *
import json
import pandas as pd


In [ ]:
CITY = "rome" # "rome" or "porto"
MEASURE = "dtw" # "dtw" or "frechet"
BUCKETING_METHOD = "loose" # Bucketing method to use

# Define parameter ranges, first value in list is parameters used for bucketing, second is parameters used for comparing trajectories with each other (using a hashed version of the trajectory) 
LAYERS_VALUES = [3, 5]  # Example layers
RESOLUTION_VALUES = [1.2, 0.3]  # Example Resolution values
DATA_SIZE = 200  # Example dataset sizes

In [ ]:
file_path = f"../../../results_true/similarity_values/{CITY}/{MEASURE}/{CITY}-{MEASURE}-{DATA_SIZE}.csv"

# Read CSV, telling pandas to take the first column as the row labels:
true_sim_matrix_df = pd.read_csv(file_path, index_col=0)

# Function to convert values to float if possible
def convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return value

# Apply the function to each cell in the DataFrame
true_sim_matrix_df = true_sim_matrix_df.map(convert_to_float)
true_sim_matrix_df = (true_sim_matrix_df + true_sim_matrix_df.T)

true_sim_matrix_df.head(10)

In [ ]:
hashed_similarities, bucket_system = generate_grid_hash_similarity_with_bucketing_hybrid(
    city=CITY, layers=LAYERS_VALUES, resolutions=RESOLUTION_VALUES, measure=MEASURE, size=DATA_SIZE, bucketing_method=BUCKETING_METHOD
    )

In [ ]:

# hashed_similarities.head(40)
#bucket system to json
with open(f"bucket_system.json", "w") as f:
    
    json.dump(bucket_system, f)

In [ ]:
hashed_similarities.head(10)

In [ ]:
index1 = "R_AAF"
index2 = "R_ABI"
value = true_sim_matrix_df.loc[index1, index2]
print(f"The value at index ({index1}, {index2}) is: {value}")

In [ ]:
total_buckets = len(bucket_system)
buckets_with_multiple = sum(1 for trajectories in bucket_system.values() if len(trajectories) > 1)
buckets_with_single = total_buckets - buckets_with_multiple
largest_bucket_size = max(len(trajectories) for trajectories in bucket_system.values())
largest_bucket = max(bucket_system, key=lambda key: len(bucket_system[key]))
smallest_bucket = min(bucket_system, key=lambda key: len(bucket_system[key]))
smallest_bucket_size = len(bucket_system[smallest_bucket])


print(f"Total Buckets: {total_buckets}")
print(f"Largest Bucket(id): {largest_bucket}")
print(f"Buckets with more than one trajectory: {buckets_with_multiple}")
print(f"Buckets with only one trajectory: {buckets_with_single}")
print(f"Largest Bucket Size: {largest_bucket_size}")
print("smallest bucket: ", smallest_bucket_size)

# Optional: Display distribution percentages
multiple_bucket_percentage = (buckets_with_multiple / total_buckets) * 100 if total_buckets > 0 else 0
single_bucket_percentage = (buckets_with_single / total_buckets) * 100 if total_buckets > 0 else 0

print(f"Percentage of buckets with more than one trajectory: {multiple_bucket_percentage:.2f}%")
print(f"Percentage of buckets with only one trajectory: {single_bucket_percentage:.2f}%")

In [ ]:
# THRESHOLD = 5
import numpy as np # type: ignore
THRESHOLDS = np.arange(1, 6.0, 1)  # Generates [0.5, 1.0, 1.5, ..., 5.5]

results = {
    "Precision": [],
    "Recall": [],
    "F1 Score": []
}



for treshold in THRESHOLDS:

    #Variables
    all_trajectory_names = list(hashed_similarities.keys()) # All trajectory names
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    precision = 0 
    recall = 0
    f1_Score = 0

    # Loop through all trajectory names
    for trajectory in all_trajectory_names:
        
        
        # Pred and ground truth
        predicted_similar = find_predicted_similar_trajectories(trajectory, bucket_system)
        ground_truth = get_nearest_neighbour_under_threshold(trajectory, treshold, true_sim_matrix_df).index.to_list()
        true_positives += calculate_true_positives(predicted_similar, ground_truth)
        false_positives += calculate_false_positives(predicted_similar, ground_truth)
        false_negatives += calculate_false_negatives(predicted_similar, ground_truth)
        # print("num predicted similar" , len(predicted_similar))
        # print("num ground truth" , len(ground_truth))

        # print(trajectory, "predicted similar: ", predicted_similar)
        # print(trajectory, "ground truth: ", ground_truth)
        # print("True positives: ", true_positives)
        # print("False positives: ", false_positives)
        # break
        

    # Calculate precision and recall
    precision = compute_bucket_system_precision(true_positives, false_positives)
    recall = compute_bucket_system_recall(true_positives, false_negatives)
    f1_score = compute_bucket_system_f1_score(precision, recall)

    results["Precision"].append(precision)
    results["Recall"].append(recall)
    results["F1 Score"].append(f1_score)
    
    
print(f"Bucket system statistics for city: {CITY}, measure: {MEASURE}, resolutions: {RESOLUTION_VALUES}, layers: {LAYERS_VALUES}, size: {DATA_SIZE}")
# Create DataFrame with thresholds as columns and metrics as row indexes
df = pd.DataFrame(results, index=[f"Threshold = {t}" for t in THRESHOLDS]).T

df